# KuaiRec + KuaiRand Lite Serving 训练与导出

这个 notebook 用于重新生成低维度、裁剪 vocab、压缩 FAISS 的轻量 serving artifacts。

推荐运行环境：
- 首选：A100 GPU + 高 RAM，性价比和稳定性适合本流程。
- H100 可用但不是必须；Two-Tower 结构较轻，瓶颈更多在 Drive I/O 和预处理。
- T4/L4 可以跑小规模验证，但不建议跑完整 200 万样本。

目标输出：
- `models/two_tower_epoch_latest.pt`
- `models/faiss_ivf_pq.index`
- `models/faiss_id_mapping.pkl`
- `data/news_vocab.pkl`
- `data/user_vocab.pkl`
- `data/preprocess_manifest.json`
- GCS: `gs://telegram-467705-recsys/artifacts/<ARTIFACT_VERSION>/...`

本流程默认不训练 Phoenix，也不发布 `item_embeddings.npy` / `news_dict.pkl` 到 serving artifacts。

## 1. 挂载 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 设置路径、压缩参数和版本

In [ ]:
import os
from pathlib import Path
from datetime import datetime

ROOT = Path('/content/drive/MyDrive/ml-services')
DATA_DIR = ROOT / 'data'
MODELS_DIR = ROOT / 'models'
SCRIPTS_DIR = ROOT / 'scripts'

KUAIREC_DIR = DATA_DIR / 'KuaiRec'
KUAIRAND_DIR = DATA_DIR / 'KuaiRand'
content_candidates = [
    DATA_DIR / 'KuaiRand-content',
    DATA_DIR / 'KuaiRand_content',
    DATA_DIR / 'KuaiRandContent',
    DATA_DIR / 'KuaiRand' / 'KuaiRand-27K',
    DATA_DIR / 'KuaiRand',
]
KUAIRAND_CONTENT_DIR = next((p for p in content_candidates if p.exists()), content_candidates[0])

GCP_PROJECT = 'telegram-467705'
GCS_BUCKET = 'telegram-467705-recsys'
ARTIFACT_VERSION = '2026-04-29_kuai_lite256'
ARTIFACT_PROFILE = 'serving-lite'

# 压缩核心参数：先用 256 维 + 30 万 item；稳定后可尝试 128 维或 50 万 item。
EMBEDDING_DIM = 256
MAX_NEWS_VOCAB_SIZE = 300_000
MIN_ITEM_FREQUENCY = 2
MAX_HISTORY = 80

# A100 高 RAM 推荐规模。若你想更快试跑，可把 train 改成 500_000。
MAX_TRAIN_SAMPLES = 2_000_000
MAX_DEV_SAMPLES = 200_000
MAX_ROWS_PER_FILE = 0
CHUNK_SIZE = 300_000
METADATA_MODE = 'none'

TWO_TOWER_EPOCHS = 8
TWO_TOWER_BATCH = 32_768
NUM_WORKERS = 8

FAISS_INDEX_TYPE = 'ivf_pq'
FAISS_IVF_PQ_M = 16
FAISS_IVF_PQ_NBITS = 8
FAISS_NPROBE = 16

os.environ.update({
    'GCP_PROJECT': GCP_PROJECT,
    'GCS_BUCKET': GCS_BUCKET,
    'ARTIFACT_VERSION': ARTIFACT_VERSION,
    'ARTIFACT_PROFILE': ARTIFACT_PROFILE,
    'DATA_DIR': str(DATA_DIR),
    'MODELS_DIR': str(MODELS_DIR),
    'KUAIREC_DIR': str(KUAIREC_DIR),
    'KUAIRAND_DIR': str(KUAIRAND_DIR),
    'KUAIRAND_CONTENT_DIR': str(KUAIRAND_CONTENT_DIR),
    'EMBEDDING_DIM': str(EMBEDDING_DIM),
    'MAX_NEWS_VOCAB_SIZE': str(MAX_NEWS_VOCAB_SIZE),
    'MIN_ITEM_FREQUENCY': str(MIN_ITEM_FREQUENCY),
    'MAX_HISTORY': str(MAX_HISTORY),
    'MAX_TRAIN_SAMPLES': str(MAX_TRAIN_SAMPLES),
    'MAX_DEV_SAMPLES': str(MAX_DEV_SAMPLES),
    'MAX_ROWS_PER_FILE': str(MAX_ROWS_PER_FILE),
    'CHUNK_SIZE': str(CHUNK_SIZE),
    'METADATA_MODE': METADATA_MODE,
    'TWO_TOWER_EPOCHS': str(TWO_TOWER_EPOCHS),
    'TWO_TOWER_BATCH': str(TWO_TOWER_BATCH),
    'NUM_WORKERS': str(NUM_WORKERS),
    'FAISS_INDEX_TYPE': FAISS_INDEX_TYPE,
    'FAISS_IVF_PQ_M': str(FAISS_IVF_PQ_M),
    'FAISS_IVF_PQ_NBITS': str(FAISS_IVF_PQ_NBITS),
    'FAISS_NPROBE': str(FAISS_NPROBE),
})

print('ROOT =', ROOT)
print('ARTIFACT_VERSION =', ARTIFACT_VERSION)
print('EMBEDDING_DIM =', EMBEDDING_DIM)
print('MAX_NEWS_VOCAB_SIZE =', MAX_NEWS_VOCAB_SIZE)
print('MAX_TRAIN_SAMPLES =', MAX_TRAIN_SAMPLES)

## 3. 检查 GPU / RAM 并给出运行建议

In [ ]:
import subprocess
import psutil

print('GPU:')
subprocess.run(['bash', '-lc', 'nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader || true'])
print()
print('RAM:')
mem = psutil.virtual_memory()
print(f'{mem.total/1024**3:.1f} GiB total, {mem.available/1024**3:.1f} GiB available')

gpu_name = subprocess.getoutput('nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null')
print()
if 'A100' in gpu_name:
    print('建议：A100 + 高 RAM 当前参数合适。预计预处理 5-15 分钟，Two-Tower 训练 3-8 分钟，FAISS 1-5 分钟。')
elif 'H100' in gpu_name:
    print('建议：H100 可以把 MAX_TRAIN_SAMPLES 提到 5_000_000，但当前 2_000_000 已足够先验证 serving-lite。')
else:
    print('建议：非 A100/H100 先把 MAX_TRAIN_SAMPLES 降到 500_000 验证流程。')


## 4. 安装依赖

In [ ]:
%cd /content/drive/MyDrive/ml-services
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.txt
!python -m pip install -q faiss-cpu google-cloud-storage psutil tqdm

## 5. 检查训练数据源

In [ ]:
from pathlib import Path

required_roots = [KUAIREC_DIR, KUAIRAND_DIR]
for root in required_roots:
    print(root, 'exists=', root.exists())
    if root.exists():
        for p in list(root.rglob('*.csv'))[:12]:
            print('  ', p.relative_to(ROOT))

expected_any = [
    'big_matrix.csv',
    'small_matrix.csv',
    'log_random_4_22_to_5_08_27k.csv',
    'log_standard_4_08_to_4_21_27k_part1.csv',
]
for name in expected_any:
    matches = list(DATA_DIR.rglob(name))
    print(name, 'matches=', len(matches), matches[:2])

## 6. 清理旧的生成产物（不会删除原始数据）

In [ ]:
from pathlib import Path

CLEAN_GENERATED_OUTPUTS = True
if CLEAN_GENERATED_OUTPUTS:
    generated_files = [
        DATA_DIR / 'news_dict.pkl',
        DATA_DIR / 'news_vocab.pkl',
        DATA_DIR / 'user_vocab.pkl',
        DATA_DIR / 'train_samples.pkl',
        DATA_DIR / 'dev_samples.pkl',
        DATA_DIR / 'item_embeddings.npy',
        DATA_DIR / 'preprocess_manifest.json',
    ]
    generated_files.extend(MODELS_DIR.glob('two_tower_epoch_*.pt'))
    generated_files.extend(MODELS_DIR.glob('phoenix_epoch_*.pt'))
    generated_files.extend(MODELS_DIR.glob('faiss_*.index'))
    generated_files.append(MODELS_DIR / 'faiss_id_mapping.pkl')

    for path in generated_files:
        try:
            if path.exists() and path.is_file():
                path.unlink()
                print('deleted', path)
        except Exception as exc:
            print('skip', path, exc)
else:
    print('skip cleanup')

## 7. 预处理并裁剪 news_vocab

In [ ]:
os.environ.update({
    'MAX_TRAIN_SAMPLES': str(MAX_TRAIN_SAMPLES),
    'MAX_DEV_SAMPLES': str(MAX_DEV_SAMPLES),
    'MAX_ROWS_PER_FILE': str(MAX_ROWS_PER_FILE),
    'CHUNK_SIZE': str(CHUNK_SIZE),
})

!python scripts/preprocess_kuaishou.py   --kuairec-dir "$KUAIREC_DIR"   --kuairand-dir "$KUAIRAND_DIR"   --kuairand-content-dir "$KUAIRAND_CONTENT_DIR"   --output-dir "$DATA_DIR"   --metadata-mode "$METADATA_MODE"   --chunk-size "$CHUNK_SIZE"   --min-history 3   --max-train-samples "$MAX_TRAIN_SAMPLES"   --max-dev-samples "$MAX_DEV_SAMPLES"   --max-rows-per-file "$MAX_ROWS_PER_FILE"   --max-news-vocab-size "$MAX_NEWS_VOCAB_SIZE"   --min-item-frequency "$MIN_ITEM_FREQUENCY"

## 8. 检查预处理结果和压缩效果

In [ ]:
import json
import pickle
from pathlib import Path

for name in ['news_dict.pkl', 'news_vocab.pkl', 'user_vocab.pkl', 'train_samples.pkl', 'dev_samples.pkl', 'preprocess_manifest.json']:
    path = DATA_DIR / name
    print(name, 'exists=', path.exists(), 'size_mb=', round(path.stat().st_size / 1024 / 1024, 2) if path.exists() else None)

manifest = json.loads((DATA_DIR / 'preprocess_manifest.json').read_text())
print(json.dumps(manifest, ensure_ascii=False, indent=2))

with open(DATA_DIR / 'train_samples.pkl', 'rb') as f:
    train_samples = pickle.load(f)
print('sample_count =', len(train_samples))
print('first_sample =', train_samples[0] if train_samples else None)

## 9. 训练低维 Two-Tower 召回模型

In [ ]:
!python scripts/train_two_tower.py   --data-dir "$DATA_DIR"   --models-dir "$MODELS_DIR"   --epochs "$TWO_TOWER_EPOCHS"   --batch-size "$TWO_TOWER_BATCH"   --embedding-dim "$EMBEDDING_DIM"   --max-history-len "$MAX_HISTORY"   --num-workers "$NUM_WORKERS"   --save-policy latest   --item-embedding-dtype float16

## 10. 构建压缩 FAISS IVF-PQ 索引

In [ ]:
!python scripts/build_faiss_index.py   --type "$FAISS_INDEX_TYPE"   --data-dir "$DATA_DIR"   --models-dir "$MODELS_DIR"   --ivf-pq-m "$FAISS_IVF_PQ_M"   --ivf-pq-nbits "$FAISS_IVF_PQ_NBITS"   --nprobe "$FAISS_NPROBE"

## 11. 校验本地轻量 artifacts

In [ ]:
from pathlib import Path

artifact_files = [
    MODELS_DIR / 'two_tower_epoch_latest.pt',
    MODELS_DIR / 'faiss_ivf_pq.index',
    MODELS_DIR / 'faiss_id_mapping.pkl',
    DATA_DIR / 'news_vocab.pkl',
    DATA_DIR / 'user_vocab.pkl',
    DATA_DIR / 'item_embeddings.npy',  # 仅本地中间产物，不会发布到 serving-lite
    DATA_DIR / 'preprocess_manifest.json',
]

total = 0
for path in artifact_files:
    size = path.stat().st_size if path.exists() else 0
    total += size
    print(path.relative_to(ROOT), 'exists=', path.exists(), 'size_mb=', round(size / 1024 / 1024, 2))
print('local_generated_total_gib=', round(total / 1024**3, 3))

## 12. 登录 Google Cloud

In [ ]:
from google.colab import auth
auth.authenticate_user()

!gcloud config set project "$GCP_PROJECT"
!gcloud auth list

## 13. 发布 serving-lite artifacts 到 GCS

In [ ]:
!python scripts/publish_artifacts.py   --bucket "$GCS_BUCKET"   --version "$ARTIFACT_VERSION"   --profile "$ARTIFACT_PROFILE"   --faiss-index-type "$FAISS_INDEX_TYPE"   --models-dir "$MODELS_DIR"   --data-dir "$DATA_DIR"

!gcloud storage ls -l "gs://$GCS_BUCKET/artifacts/$ARTIFACT_VERSION/**" | tail -30

## 14. 估算发布后的 GCS 体积

In [ ]:
!gcloud storage du -s "gs://$GCS_BUCKET/artifacts/$ARTIFACT_VERSION/**"

## 15. 后续 Cloud Run 切换提醒

In [ ]:
print('不要直接运行旧的 Cloud Run 切换单元。')
print('原因：当前线上 ml-services 代码仍把 Phoenix / item_embeddings 视作完整包路径的一部分。')
print('下一步需要先部署支持 serving-lite 的服务端代码，再把 ARTIFACT_VERSION 切到:', ARTIFACT_VERSION)
print('推荐 Cloud Run 配置：minScale=0, 4 vCPU / 16Gi, PRELOAD_MODELS_ON_STARTUP=false。')